In [1]:
import glob
print(glob.glob("/kaggle/input/**/crnn_id_best.pth", recursive=True))
print(glob.glob("/kaggle/input/**/surat01.png", recursive=True))
print(glob.glob("/kaggle/input/**/labels.csv", recursive=True))

['/kaggle/input/datasets/akundummy12/ocr-tes-surat/crnn_id_best.pth']
['/kaggle/input/datasets/akundummy12/ocr-tes-surat/contoh_surat/contoh_surat/pages/surat01.png']
['/kaggle/input/datasets/akundummy12/ocr-tes-surat/contoh_surat/contoh_surat/real_lines_full/labels.csv', '/kaggle/input/datasets/akundummy12/ocr-tes-surat/contoh_surat/contoh_surat/real_lines/labels.csv']


In [2]:
import os, csv, string
import numpy as np, torch, torch.nn as nn
from PIL import Image, ImageOps

device = "cuda" if torch.cuda.is_available() else "cpu"
BASE = "/kaggle/input/datasets/akundummy12/ocr-tes-surat"
CKPT = BASE + "/crnn_id_best.pth"
DATA = BASE + "/contoh_surat/contoh_surat"

IMG_H = 32
CHARS = string.ascii_letters + string.digits + ".,:;-/()'\"&%#@!?+ "
BLANK = 0
idx2char = {i + 1: c for i, c in enumerate(CHARS)}

class CRNN(nn.Module):
    def __init__(self, num_classes, hidden=256):
        super().__init__()
        def block(i, o):
            return [nn.Conv2d(i, o, 3, 1, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True)]
        self.cnn = nn.Sequential(
            *block(1, 64),    nn.MaxPool2d(2, 2),
            *block(64, 128),  nn.MaxPool2d(2, 2),
            *block(128, 256), *block(256, 256), nn.MaxPool2d((2, 1)),
            *block(256, 512), *block(512, 512), nn.MaxPool2d((2, 1)),
            nn.Conv2d(512, 512, (2, 1), bias=False), nn.BatchNorm2d(512), nn.ReLU(True),
        )
        self.rnn = nn.LSTM(512, hidden, num_layers=2, bidirectional=True, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden * 2, num_classes)
    def forward(self, x):
        f = self.cnn(x).squeeze(2).permute(0, 2, 1)
        f, _ = self.rnn(f)
        return self.fc(f)

def greedy_decode(logits):
    p = logits.argmax(2)[0].cpu().numpy()
    s, prev = [], 0
    for k in p:
        if k != prev and k != BLANK:
            s.append(idx2char[int(k)])
        prev = k
    return "".join(s)

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def prep_image(img):
    img = ImageOps.autocontrast(img.convert("L"))
    w, h = img.size
    new_w = min(max(int(w * IMG_H / h), 32), 2000)
    img = img.resize((new_w, IMG_H), Image.BILINEAR)
    arr = (np.asarray(img, dtype=np.float32) / 255.0 - 0.5) / 0.5
    return torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)

ckpt = torch.load(CKPT, map_location=device)
assert ckpt["chars"] == CHARS, "Charset berbeda!"
model = CRNN(len(CHARS) + 1).to(device)
model.load_state_dict(ckpt["model"]); model.eval()
print("Model dimuat. Epoch:", ckpt.get("epoch"), "| device:", device)

@torch.no_grad()
def tes_folder(name, n_show=8):
    folder = f"{DATA}/{name}"
    rows = list(csv.DictReader(open(f"{folder}/labels.csv", encoding="utf-8")))
    hasil, c_err, c_tot, exact = [], 0, 0, 0
    for r in rows:
        pred = greedy_decode(model(prep_image(Image.open(f"{folder}/{r['filename']}")).to(device)))
        gt = r["text"]
        e = edit_distance(pred, gt)
        c_err += e; c_tot += len(gt); exact += (pred == gt)
        hasil.append((e, r["filename"], gt, pred))
    print(f"\n=== {name}: {len(rows)} baris | CER {c_err / c_tot * 100:.2f}% | "
          f"persis benar {exact / len(rows) * 100:.1f}% ===")
    print(f"{n_show} baris terburuk:")
    for e, fn, gt, pred in sorted(hasil, reverse=True)[:n_show]:
        print(f"  {fn}  (salah {e} huruf)\n     GT  : {gt}\n     Pred: {pred}")

tes_folder("real_lines")
tes_folder("real_lines_full")

Model dimuat. Epoch: 18 | device: cuda

=== real_lines: 175 baris | CER 0.13% | persis benar 97.7% ===
8 baris terburuk:
  surat05_b18_1.png  (salah 3 huruf)
     GT  : Penanggung Jawab : Pratama Hasan, S.Kom.
     Pred: Penanggung JawabPratama Hasan, S.Kom.
  surat04_b13_1.png  (salah 1 huruf)
     GT  : NIP : 19681014 200103 1 878
     Pred: NIP: 19681014 200103 1 878
  surat04_b09_1.png  (salah 1 huruf)
     GT  : NIP : 19710407 199911 2 876
     Pred: NIP: 19710407 199911 2 876
  surat03_b04_1.png  (salah 1 huruf)
     GT  : Telepon (0778) 123456, Email:
     Pred: Telepon (0778) 123456, Email: 
  surat05_b27_1.png  (salah 0 huruf)
     GT  : 2. Arsip
     Pred: 2. Arsip
  surat05_b26_1.png  (salah 0 huruf)
     GT  : 1. Kepala Seksi Teknik
     Pred: 1. Kepala Seksi Teknik
  surat05_b25_1.png  (salah 0 huruf)
     GT  : Tembusan:
     Pred: Tembusan:
  surat05_b24_1.png  (salah 0 huruf)
     GT  : NIP. 19750424 199908 1 716
     Pred: NIP. 19750424 199908 1 716

=== real_lines_ful